## Linguistic Analysis *Confessio Patricii*

**Course:** Computational Philology
**Source text:** *Confessio* of Saint Patrick (Latin, 5th c. AD)
**Pipeline stages:**

1. **Text Segmentation**  1.1 Tokenization, 1.2 Normalization
2. **Text Verticalization**  2.1 Lemmatization, 2.2 POS Tagging, 2.3 Morphological Analysis, 2.4 Syntactic Analysis
3. **ID Attribution** 


## 0. Setup


In [2]:
! pip install conllu

In [41]:
# Standard library
import re
import json
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
from conllu import parse, parse_incr
import os

# Paths (adjust if running outside this folder structure)
DATA_DIR = Path(".")
RAW_TEXT_PATH = DATA_DIR / "0.Confessio_Latin.txt"
CONLLU_PATH = DATA_DIR / "Confessio_Latin_corrected_v2.conllu"  

pd.set_option("display.max_colwidth", 60)
print("Setup complete.")


Setup complete.


In [9]:
print (os.getcwd())
print(os.listdir())

/Users/aima/Desktop/Practice/GitHub/research-computational_philology/3-notebooks
['Confessio_Latin_corrected_v2.conllu', '.DS_Store', 'Confessio_Linguistic_Analysis_v2.ipynb', '0.Confessio_Latin.txt', 'confessio analysis_V1.ipynb', 'Confessio_English.txt']


## 1. Text Segmentation

Text segmentation is the first stage of the pipeline:   
-Tokenization     
-Normalization      

### 1.1 Tokenization

Tokenization splits the text into sentences and word-level tokens. 
Spliting was already performed by **UDPipe** (the `latin-ittb-ud` model) when the `.conllu` file was
generated and manually corrected by **Arborator Grew** application


In [36]:
# Load raw text
raw_text = RAW_TEXT_PATH.read_text(encoding="utf-8")
print(f"Raw text length: {len(raw_text)} characters")
print("---")
print(raw_text[:300], "...")


Raw text length: 2529 characters
---
Ego Patricius peccator rusticissimus et minimus omnium fidelium et contemptibilissimus apud plurimos patrem habui Calpornium diaconum filium quendam Potiti presbyteri, qui fuit uico bannauem taburniae;
uillulam enim prope habuit, ubi ego capturam dedi.
Annorum eram tunc fere sedecim.
Deum enim uerum ...


In [ ]:
# Check sentence boundaries of conllu file.

def naive_sentence_split(text):
    # Split on '.', ';', ':' followed by whitespace, keeping the punctuation.
    text = text.replace("\n", " ")
    sentences =  re.split(r"(?<=[.;])\s+",text.strip())
    return [s.strip() for s in sentences if s.strip()]

raw_sentences = naive_sentence_split(raw_text)
print(f"Base sentence count: {len(raw_sentences)}")
for i, s in enumerate(raw_sentences[:], start=1):
    print(f"[{i}] {s}")


Base sentence count: 12
[1] Ego Patricius peccator rusticissimus et minimus omnium fidelium et contemptibilissimus apud plurimos patrem habui Calpornium diaconum filium quendam Potiti presbyteri, qui fuit uico bannauem taburniae;
[2] uillulam enim prope habuit, ubi ego capturam dedi.
[3] Annorum eram tunc fere sedecim.
[4] Deum enim uerum ignorabam et Hiberione in captiuitate adductus sum cum tot milia hominum, secundum merita nostra, quia a Deo recessimus et praecepta eius non custodiuimus et sacerdotibus nostris non oboedientes fuimus, qui nos nostram salutem admonebant;
[5] et Dominus induxit super nos iram animationis suae et dispersit nos in gentibus multis etiam usque ad ultimum terrae, ubi nunc paruitas mea esse uidetur inter alienigenas, et ibi Dominus aperuit sensum incredulitatis meae, ut uel sero rememorarem delicta mea et ut conuerterem toto corde ad Dominum Deum meum, qui respexit humilitatem meam et misertus est adolescentiae et ignorantiae meae et custodiuit me antequam 

In [ ]:
# Check tokenizer  (splits on whitespace and punctuation).
# Used only to compare token counts 
TOKEN_PATTERN = re.compile(r"[A-Za-zÀ-ÿ]+|[.;:,]")

def naive_tokenize(sentence: str) -> list[str]:
    return TOKEN_PATTERN.findall(sentence)

sample_tokens = naive_tokenize(raw_sentences[0])
print(f"Tokens in sentence 1 (base): {len(sample_tokens)}")
print(sample_tokens)


Tokens in sentence 1 (naive): 27
['Ego', 'Patricius', 'peccator', 'rusticissimus', 'et', 'minimus', 'omnium', 'fidelium', 'et', 'contemptibilissimus', 'apud', 'plurimos', 'patrem', 'habui', 'Calpornium', 'diaconum', 'filium', 'quendam', 'Potiti', 'presbyteri', ',', 'qui', 'fuit', 'uico', 'bannauem', 'taburniae', ';']


### 1.2 Normalization

Normalization maps different attestations of the same form to a more comparable
representation.    
For Latin, the most relevant normalization is **u/v and i/j orthographic variation**
(e.g. `uico` vs `vico`, `iam` vs `jam`).     
Orthographic normalization was considered but not applied, since the source edition already follows a regularized classicizing Latin orthography   
UDPipe's `latin-ittb` model expects the **u/v-consonantal, classicizing** convention, so text normalize accordingly *before* parsing.


In [38]:
#Create a change log
#Creat a working copy (the original text is not changed.)
#List of rules(each rule consists of: pattern, replacement, description)
def normalize_latin(text):
    log = []
    normalized = text
    rules = [
        # (pattern, replacement, description)  currently empty
    ]

    for pattern, replacement, _desc in rules:
        matches = re.findall(pattern, normalized)
        if matches:
            log.append((pattern, replacement))
        normalized = re.sub(pattern, replacement, normalized)

    return normalized, log

normalized_text, normalization_log = normalize_latin(raw_text)
print("Normalization rules applied:", len(normalization_log))
print("Text unchanged:", normalized_text == raw_text)


Normalization rules applied: 0
Text unchanged: True


## 2. Text Verticalization

Verticalization turns the horizontal text stream into a **one token per line** table.      
The structure of the **CoNLL-U** format.    
Load the .conllu file produced by UDPipe (+ manual correction in Arborator Grew).    
Parse it into a flat, per-token table that holds every annotation layer side by side:

| # | Layer | CoNLL-U column | Produced by |
|---|---|---|---|
| 2.1 | Lemmatization | `LEMMA` | UDPipe |
| 2.2 | POS Tagging | `UPOS`, `XPOS` | UDPipe |
| 2.3 | Morphological Analysis | `FEATS` | UDPipe |
| 2.4 | Syntactic Analysis | `HEAD`, `DEPREL`, `DEPS` | UDPipe (+ manual correction in Arborator Grew) |


In [42]:
# Parse the CoNLL-U file into a list of TokenList objects (one per sentence)
with open(CONLLU_PATH, "r", encoding="utf-8") as f:
    sentences = list(parse_incr(f))

print(f"Number of sentences: {len(sentences)}")
print(f"Tokens in sentence 1: {len(sentences[0])}")
print(f"Sentence 1 text: {sentences[0].metadata.get('text')}")


Number of sentences: 12
Tokens in sentence 1: 27
Sentence 1 text: Ego Patricius peccator rusticissimus et minimus omnium fidelium et contemptibilissimus apud plurimos patrem habui Calpornium diaconum filium quendam Potiti presbyteri, qui fuit uico bannauem taburniae;


In [44]:
print(dir(sentences[0]))
print(type(sentences[0]))

['__add__', '__annotations__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__imul__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__mul__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__reversed__', '__rmul__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_dict_to_token_and_set_defaults', 'append', 'clear', 'copy', 'count', 'default_fields', 'extend', 'filter', 'head_to_token', 'index', 'insert', 'metadata', 'pop', 'remove', 'reverse', 'serialize', 'sort', 'to_tree']
<class 'conllu.models.TokenList'>


In [ ]:
# Flatten all sentences into a single per-token DataFrame.
# Each row = one token.   
# Each column = one CoNLL-U annotation layer.

records = []
for sent_idx, sent in enumerate(sentences, start=1):
    sent_id = sent.metadata.get("sent_id", sent_idx)
    for tok in sent:
        # Skip multiword tokens(keep only simple tokens)
        if isinstance(tok["id"], tuple):
            continue
        records.append({
            "sent_id": sent_id,
            "id": tok["id"],
            "form": tok["form"],
            "lemma": tok["lemma"],
            "upos": tok["upos"],
            "xpos": tok["xpos"],
            "feats": tok["feats"],
            "head": tok["head"],
            "deprel": tok["deprel"],
            "deps": tok["deps"],
            "misc": tok["misc"],
        })

df = pd.DataFrame(records)
print(f"Total tokens: {len(df)}")
df.head(10)


Total tokens: 424


,sent_id,id,form,lemma,upos,xpos,feats,head,deprel,deps,misc
0,1,1,Ego,ego,PRON,F1|grn1|casA|gen1,"{'Case': 'Nom', 'Number': 'Sing', 'Person': '1', 'PronTy...",14,nsubj,None,"{'InflClass': 'LatAnom', 'TokenRange': '0:3'}"
1,1,2,Patricius,patricius,PROPN,B1|grn1|casA|gen1,"{'Case': 'Nom', 'Gender': 'Masc', 'Number': 'Sing'}",1,appos,None,"{'InflClass': 'IndEurO', 'NameType': 'Giv', 'TokenRange'..."
2,1,3,peccator,peccator,NOUN,C1|grn1|casA|gen1,"{'Case': 'Nom', 'Gender': 'Masc', 'Number': 'Sing'}",2,appos,None,"{'InflClass': 'IndEurX', 'TokenRange': '14:22'}"
3,1,4,rusticissimus,rusticus,ADJ,B1|grn3|casA|gen1,"{'Case': 'Nom', 'Degree': 'Abs', 'Gender': 'Masc', 'Numb...",3,amod,None,"{'InflClass': 'IndEurO', 'TokenRange': '23:36'}"
4,1,5,et,et,CCONJ,O4,None,6,cc,None,{'TokenRange': '37:39'}
5,1,6,minimus,paruus,ADJ,B1|grn3|casA|gen1,"{'Case': 'Nom', 'Degree': 'Abs', 'Gender': 'Masc', 'Numb...",4,conj,None,"{'InflClass': 'IndEurO', 'TokenRange': '40:47'}"
6,1,7,omnium,omnis,DET,C1|grn1|casK|gen3,"{'Case': 'Gen', 'Gender': 'Neut', 'Number': 'Plur', 'Pro...",8,det,None,"{'InflClass': 'IndEurI', 'TokenRange': '48:54'}"
7,1,8,fidelium,fidelis,ADJ,C1|grn1|casK|gen1,"{'Case': 'Gen', 'Gender': 'Masc', 'Number': 'Plur'}",3,nmod,None,"{'InflClass': 'IndEurI', 'TokenRange': '55:63'}"
8,1,9,et,et,CCONJ,O4,None,10,cc,None,{'TokenRange': '64:66'}
9,1,10,contemptibilissimus,contemptibilis,ADJ,B1|grn3|casA|gen1,"{'Case': 'Nom', 'Degree': 'Abs', 'Gender': 'Masc', 'Numb...",4,conj,None,"{'InflClass': 'IndEurO', 'TokenRange': '67:86'}"


### 2.1 Lemmatization

The `LEMMA` column gives the dictionary (citation) form for each token, independent of
its inflected form in the text.    
Lets us check whether the lemmatizer behaved consistently e.g. all forms of *habeo* (`habui`, `habuit`...) should map to the same lemma.


In [46]:
# Most frequent lemmas in the text (function words excluded for readability)
lemma_counts = df["lemma"].value_counts().head(15)
lemma_counts


lemma
et         37
,          28
sum        19
qui        11
deus       10
omnis       9
ego         8
in          7
.           7
meus        7
pater       6
ut          6
dominus     5
quia        4
:           4
Name: count, dtype: int64

In [ ]:
# How many distinct FORM surface realizations map to each LEMMA?

form_per_lemma = df.groupby("lemma")["form"].nunique().sort_values(ascending=False).head(10)
form_per_lemma


lemma
sum       10
omnis      6
qui        5
deus       4
ego        4
meus       4
pater      4
noster     3
is         3
ipse       3
Name: form, dtype: int64

### 2.2 POS Tagging

`UPOS` is the **Universal POS tag** (language-independent: `VERB`, `NOUN`, `ADJ`, ...).     
`XPOS` is the **language/treebank-specific tag** — here, the **ITTB** tagset used for the
`latin-ittb` UDPipe model.


In [48]:
upos_distribution = df["upos"].value_counts()
upos_distribution


upos
NOUN     87
VERB     58
CCONJ    46
PUNCT    43
DET      40
ADP      33
PRON     28
ADJ      20
AUX      19
ADV      18
SCONJ    12
PART     10
PROPN     8
NUM       2
Name: count, dtype: int64

In [ ]:
# UPOS distribution as a percentage.
upos_pct = (upos_distribution / len(df) * 100).round(2)
upos_pct.rename("percent(%)").to_frame()


,percent(%)
upos,
NOUN,20.52
VERB,13.68
CCONJ,10.85
PUNCT,10.14
DET,9.43
ADP,7.78
PRON,6.60
ADJ,4.72
AUX,4.48


### 2.3 Morphological Analysis

`FEATS` stores morphological feature-value pairs (`Case=Nom|Number=Sing|...`)    
Following the **UD Universal Features** inventory.         
Parse this column into individual feature columns to enable querying (e.g. *"all perfect-tense verbs"*).


In [ ]:
def parse_feats(feats):
    if not feats:
        return {}
    return dict(feats)
df["feats_dict"] = df["feats"].apply(parse_feats)
# Expand the most relevant morphological features into their own columns
for feat_name in ["Case", "Number", "Gender", "Tense", "Mood", "Person", "VerbForm", "Voice"]:
    df[feat_name] = df["feats_dict"].apply(lambda d: d.get(feat_name))

df[["form", "lemma", "upos", "Case", "Number", "Tense", "Mood", "VerbForm"]].head(10)


,form,lemma,upos,Case,Number,Tense,Mood,VerbForm
0,Ego,ego,PRON,Nom,Sing,None,None,None
1,Patricius,patricius,PROPN,Nom,Sing,None,None,None
2,peccator,peccator,NOUN,Nom,Sing,None,None,None
3,rusticissimus,rusticus,ADJ,Nom,Sing,None,None,None
4,et,et,CCONJ,None,None,None,None,None
5,minimus,paruus,ADJ,Nom,Sing,None,None,None
6,omnium,omnis,DET,Gen,Plur,None,None,None
7,fidelium,fidelis,ADJ,Gen,Plur,None,None,None
8,et,et,CCONJ,None,None,None,None,None
9,contemptibilissimus,contemptibilis,ADJ,Nom,Sing,None,None,None


In [54]:
# All finite verbs in the Imperfect tense acting as the root of their clause.
imperfect_roots = df[(df["upos"] == "VERB") & (df["Tense"] == "Imp") & (df["deprel"] == "root")]
imperfect_roots[["sent_id", "form", "lemma", "Tense", "Mood", "deprel"]]


,sent_id,form,lemma,Tense,Mood,deprel


### 2.4 Syntactic Analysis

`HEAD` gives the token ID of the syntactic governor within the sentence (`0` = root).   
`DEPREL` gives the Universal Dependencies relation label (`root`, `nsubj`, `conj`, ...).   
`DEPS` holds Enhanced UD dependencies, usually empty (`_`) unless manually enriched.    



In [55]:
deprel_distribution = df["deprel"].value_counts()
deprel_distribution


deprel
cc             44
punct          43
conj           40
case           33
obj            31
obl            30
det            29
nmod           23
nsubj          16
advmod         14
mark           13
root           12
appos          11
acl:relcl       9
amod            8
obl:arg         8
advcl           7
acl             7
cop             6
discourse       5
xcomp           5
aux:pass        5
iobj            3
advmod:neg      3
nsubj:pass      3
advmod:tmod     3
ccomp           2
advmod:lmod     2
flat            2
nummod          1
aux             1
advmod:emph     1
compound        1
amod:neg        1
parataxis       1
csubj           1
Name: count, dtype: int64

In [56]:
# Coordination chains (deprel == 'conj') per sentence directly relevant to the stylistic observation in the notes      
# ("Patrick strings six verbs under one ROOT via conj in sentence 5")
conj_per_sentence = df[df["deprel"] == "conj"].groupby("sent_id").size().sort_values(ascending=False)
conj_per_sentence.head(10)


sent_id
9        15
5        11
4         3
1         2
11        2
7         2
6         2
8         2
12_13     1
dtype: int64

## 3. ID Attribution
Two ID layers are combined here:

1. **Local ID** the CoNLL-U `(sent_id, id)` pair, unique within this file.
2. **Global Token ID** a single running integer across the whole document, useful for
   joining this table with external resources (e.g. a Arborator Grew export, or a stand-off
   annotation layer) and for citing a precise token position in the philological commentary.

 `TokenRange`also found in the `MISC` column (character offsets in the
original `.txt` file), which trace every annotated token back to its exact
position in the source.


In [57]:
def extract_token_range(misc):
    # Extract the TokenRange=start:end value from the MISC field, if present.
    if not misc:
        return None
    # `conllu` parses MISC into a dict already when possible
    if isinstance(misc, dict):
        return misc.get("TokenRange")
    return None

df["token_range"] = df["misc"].apply(extract_token_range)

# Global, document-level running ID — independent from sentence boundaries
df["global_id"] = range(1, len(df) + 1)

# Composite local ID, e.g. "3.14" = sentence 3, token 14
df["local_id"] = df.apply(lambda r: f"{r['sent_id']}.{r['id']}", axis=1)

df[["global_id", "local_id", "sent_id", "id", "form", "token_range"]].head(10)


,global_id,local_id,sent_id,id,form,token_range
0,1,1.1,1,1,Ego,0:3
1,2,1.2,1,2,Patricius,4:13
2,3,1.3,1,3,peccator,14:22
3,4,1.4,1,4,rusticissimus,23:36
4,5,1.5,1,5,et,37:39
5,6,1.6,1,6,minimus,40:47
6,7,1.7,1,7,omnium,48:54
7,8,1.8,1,8,fidelium,55:63
8,9,1.9,1,9,et,64:66
9,10,1.10,1,10,contemptibilissimus,67:86


In [58]:
# Verify ID integrity: no duplicate local IDs, no gaps in global IDs
assert df["local_id"].is_unique, "Duplicate local IDs found!"
assert df["global_id"].is_unique, "Duplicate global IDs found!"
print("ID integrity check passed: all local_id and global_id values are unique.")
print(f"Document spans {df['sent_id'].nunique()} sentences and {len(df)} tokens.")


ID integrity check passed: all local_id and global_id values are unique.
Document spans 12 sentences and 424 tokens.


## 4. Quantitative Metrics

Patrick has a strange writing style, long sentences, an overuse of "and" words (cojunctions), and unusual grammatical constructions.     
Section 4 turns this feeling into numbers, it counts:    
How many words are in each sentence,
How "deep" the sentence's grammatical tree is (how many levels of nesting it has).    
How many words are connected by "and" in a row.  
How many times the word "ut" (so that) is followed by an infinitive instead of the correct classical form.      
This section computes, directly on the single corrected .conllu file loaded in Section 2 (`sentences`).

### 4.1 Sentence-level metrics

Compute the metrics for every sentence of the **corrected** file:     
- Sentence length
- Dependency tree depth
- Length of `conj` coordination chains attached to the main `root`.

In [59]:
def build_head_map(sentence):
    # id -> head, restricted to simple (non-multiword) tokens.
    return {tok["id"]: tok["head"] for tok in sentence if not isinstance(tok["id"], tuple)}

def token_depth(token_id, head_map):
    # Depth of a token = number of edges to the root (root itself = depth 0).
    depth = 0
    current = token_id
    visited = set()
    while head_map.get(current, 0) != 0:
        if current in visited:  # safety guard against cyclic/malformed trees
            break
        visited.add(current)
        current = head_map[current]
        depth += 1
    return depth

def longest_conj_chain(sentence):
    # Longest run of consecutive conj-conj-conj... edges anywhere in the sentence.
    # This captures coordination chains regardless of which node they ultimately
    # attach to (root, ccomp, acl...), unlike a root-only count.
    by_id = {t["id"]: t for t in sentence if not isinstance(t["id"], tuple)}
    best = 0
    for tok in by_id.values():
        if tok["deprel"] != "conj":
            continue
        length = 1
        current = tok
        visited = {current["id"]}
        while True:
            parent = by_id.get(current["head"])
            if parent is None or parent["deprel"] != "conj" or parent["id"] in visited:
                break
            length += 1
            visited.add(parent["id"])
            current = parent
        best = max(best, length)
    return best

sentence_metrics = []
for sent_idx, sent in enumerate(sentences, start=1):
    sent_id = sent.metadata.get("sent_id", sent_idx)
    simple_tokens = [t for t in sent if not isinstance(t["id"], tuple)]
    head_map = build_head_map(sent)

    n_tokens = len(simple_tokens)
    depths = [token_depth(t["id"], head_map) for t in simple_tokens]
    max_depth = max(depths) if depths else 0

    total_conj = sum(1 for t in simple_tokens if t["deprel"] == "conj")
    longest_chain = longest_conj_chain(sent)

    sentence_metrics.append({
        "sent_id": sent_id,
        "n_tokens": n_tokens,
        "tree_depth": max_depth,
        "total_conj": total_conj,
        "longest_conj_chain": longest_chain,
    })

metrics_df = pd.DataFrame(sentence_metrics)
metrics_df


,sent_id,n_tokens,tree_depth,total_conj,longest_conj_chain
0,1,27,10,2,1
1,2,10,3,0,0
2,3,6,2,0,0
3,4,41,6,3,1
4,5,90,11,11,1
5,6,26,7,2,1
6,7,26,5,2,1
7,8,34,5,2,2
8,9,132,11,15,5
9,10,6,2,0,0


In [60]:
# Descriptive statistics across the whole document
metrics_df[["n_tokens", "tree_depth", "total_conj", "longest_conj_chain"]].describe().round(2)


,n_tokens,tree_depth,total_conj,longest_conj_chain
count,12.00,12.00,12.00,12.00
mean,35.33,5.67,3.33,1.17
std,38.10,3.39,4.70,1.34
min,6.00,2.00,0.00,0.00
25%,12.25,3.00,0.75,0.75
50%,26.00,5.00,2.00,1.00
75%,35.75,7.75,2.25,1.00
max,132.00,11.00,15.00,5.00


Note: Sentence 9, dedicated to the Trinity.   
Is the longest sentence in the text, containing 132 words.    
It also has the highest number of tokens and the greatest dependency-tree depth.    
At first glance, the sentence appears to be built around a long chain of coordinated verbs linked by repeated instances of "and".    

The quantitative analysis confirmed the presence of this extended verbal chain, but it also revealed a more complex underlying structure.     
Rather than being attached directly to the root of the sentence, the sequence of verbs is embedded within subordinate constructions represented by ccomp and acl relations.     
In other words, what appears during close reading to be a straightforward series of parallel verbs is actually nested inside a deeper syntactic layer.     

This finding is a useful example of how computational analysis can complement traditional close reading.     
Manual observation correctly identified the most salient stylistic feature of the passage the unusually long sequence of coordinated verbs.    
However, only the dependency analysis made it possible to determine how that sequence is structurally organized within the sentence.    



### Sources
- UDPipe / `latin-ittb` model: https://lindat.mff.cuni.cz/services/udpipe
- Universal Dependencies (Latin): https://universaldependencies.org/la/
- Arborator Grew: https://arborator.grew.fr
- Guidelines for the Syntactic Annotation of Latin Treebanks (PDT-based): `guidelines_latin.pdf`
